# 🏆 Modul 06 Hands-on: Real Project - End-to-End Loan Default Prediction

Pada proyek akhir ini, Anda akan membangun sistem Machine Learning perbankan end-to-end:
1. **Simulasi Dataset Kredit Realistis** (Campuran numerik & kategorikal).
2. **ColumnTransformer Pipeline** berstandar industri.
3. **Model Tournament**: Membandingkan 3 model dengan Cross-Validation.
4. **Production Inference Function**.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib, os

# 1. Buat Dataset Aplikasi Kredit Sintetis (1.500 nasabah)
np.random.seed(42)
N = 1500
df = pd.DataFrame({
    "Age": np.random.randint(21, 65, N),
    "Income": np.random.exponential(scale=50000, size=N) + 15000,
    "LoanAmount": np.random.uniform(5000, 40000, N),
    "CreditScore": np.random.normal(650, 70, N).clip(300, 850),
    "EmploymentType": np.random.choice(["Salaried", "Self-Employed", "Unemployed"], size=N, p=[0.7, 0.25, 0.05]),
    "HomeOwnership": np.random.choice(["RENT", "MORTGAGE", "OWN"], size=N, p=[0.5, 0.4, 0.1]),
})

# Logika default realistis
prob_default = 1 / (1 + np.exp(-(-0.04*(df["CreditScore"]-600) + 0.00003*df["LoanAmount"] - 0.00002*df["Income"] + (df["EmploymentType"]=="Unemployed")*1.5)))
df["Default"] = (np.random.rand(N) < prob_default).astype(int)

print(f"Total data: {len(df)} nasabah. Rasio Default: {df['Default'].mean()*100:.1f}%")
df.head()

## 2. Membangun ColumnTransformer Pipeline

In [ ]:
X = df.drop(columns=["Default"])
y = df["Default"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = ["Age", "Income", "LoanAmount", "CreditScore"]
cat_cols = ["EmploymentType", "HomeOwnership"]

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])
print("Preprocessor ColumnTransformer siap!")

## 3. Model Tournament: Menemukan Model Terbaik

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=500),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", clf)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name:20s} -> CV ROC-AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

## 4. Final Packaging & Production Inference Function

In [ ]:
final_pipeline = Pipeline([("prep", preprocessor), ("model", GradientBoostingClassifier(random_state=42))])
final_pipeline.fit(X_train, y_train)

# Simpan pipeline utuh ke disk
MODEL_FILE = "credit_risk_pipeline.joblib"
joblib.dump(final_pipeline, MODEL_FILE)
print(f"Pipeline produksi berhasil disimpan ke: {MODEL_FILE}")

# Fungsi inferensi produksi
def evaluate_applicant(applicant_data: dict):
    loaded = joblib.load(MODEL_FILE)
    input_df = pd.DataFrame([applicant_data])
    prob = loaded.predict_proba(input_df)[0, 1]
    status = "DITOLAK (Risiko Default Tinggi)" if prob > 0.35 else "DISETUJUI (Risiko Rendah)"
    return {"Default Probability": f"{prob*100:.2f}%", "Keputusan": status}

# Uji coba pelamar baru
pelamar_1 = {"Age": 32, "Income": 85000, "LoanAmount": 15000, "CreditScore": 740, "EmploymentType": "Salaried", "HomeOwnership": "MORTGAGE"}
pelamar_2 = {"Age": 23, "Income": 18000, "LoanAmount": 30000, "CreditScore": 520, "EmploymentType": "Unemployed", "HomeOwnership": "RENT"}

print("Hasil Evaluasi Nasabah 1:", evaluate_applicant(pelamar_1))
print("Hasil Evaluasi Nasabah 2:", evaluate_applicant(pelamar_2))
print("\nSelamat! Anda telah membangun sistem Machine Learning end-to-end kelas produksi!")